In [1]:
import hashlib
import os
import platform
import uuid

# 修改为自己的学号
STUDENT_ID = 20244080304

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]

RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]       {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")

[Conda 环境] exp01_env
[学号]       20244080304
[机器指纹] 5fc15abbc5865ff7
[RUN_ID]   20244080304-5fc15abbc5865ff7


## 一、正则表达式练习

In [2]:
import re

In [3]:
def is_alnum_only(s):
    return re.fullmatch(r"[A-Za-z0-9]+", s) is not None


for s in ["abc123", "ABC", "123", "abc 123", "ab-c", "中文"]:
    print(repr(s), is_alnum_only(s))

'abc123' True
'ABC' True
'123' True
'abc 123' False
'ab-c' False
'中文' False


In [4]:
def match_a_then_bs(s):
    return re.fullmatch(r"ab*", s) is not None


for s in ["a", "ab", "abbb", "b", "aab", "ac"]:
    print(repr(s), match_a_then_bs(s))

'a' True
'ab' True
'abbb' True
'b' False
'aab' False
'ac' False


In [5]:
def match_a_then_at_least_one_b(s):
    return re.fullmatch(r"ab+", s) is not None


for s in ["a", "ab", "abbb", "b", "aab"]:
    print(repr(s), match_a_then_at_least_one_b(s))

'a' False
'ab' True
'abbb' True
'b' False
'aab' False


In [6]:
def starts_with_a_ends_with_b(s):
    return re.fullmatch(r"a.*b", s) is not None


for s in ["ab", "a123b", "a b b", "a", "b", "acb"]:
    print(repr(s), starts_with_a_ends_with_b(s))

'ab' True
'a123b' True
'a b b' True
'a' False
'b' False
'acb' True


In [7]:
def strip_ip_zeros(ip):
    return re.sub(r"\b0+(\d)", r"\1", ip)


for ip in ["192.168.001.010", "10.000.0.01", "000.000.000.000", "172.16.254.1"]:
    print(ip, "->", strip_ip_zeros(ip))

192.168.001.010 -> 192.168.1.10
10.000.0.01 -> 10.0.0.1
000.000.000.000 -> 0.0.0.0
172.16.254.1 -> 172.16.254.1


In [8]:
def swap_space_underscore(text):
    return re.sub(r"[ _]", lambda m: "_" if m.group() == " " else " ", text)


for s in ["hello world", "hello_world", "a b_c d_e"]:
    print(repr(s), "->", repr(swap_space_underscore(s)))

'hello world' -> 'hello_world'
'hello_world' -> 'hello world'
'a b_c d_e' -> 'a_b c_d e'


In [9]:
def to_dd_mm_yyyy(text):
    return re.sub(r"(\d{4})-(\d{2})-(\d{2})", r"\3-\2-\1", text)


for s in ["2024-03-05", "最后期限 2025-12-31，请勿拖延"]:
    print(s, "->", to_dd_mm_yyyy(s))

2024-03-05 -> 05-03-2024
最后期限 2025-12-31，请勿拖延 -> 最后期限 31-12-2025，请勿拖延


In [10]:
def ae_words(text):
    return re.findall(r"\b[aeAE]\w*", text)


for s in ["an apple and an elephant", "Every answer is easy", "banana"]:
    print(repr(s), ae_words(s))

'an apple and an elephant' ['an', 'apple', 'and', 'an', 'elephant']
'Every answer is easy' ['Every', 'answer', 'easy']
'banana' []


In [11]:
def abbreviate_road(text):
    return re.sub(r"\bRoad\b", "Rd", text)


for s in ["Main Road", "Roadside Road", "Baker Road, London Road"]:
    print(s, "->", abbreviate_road(s))

Main Road -> Main Rd
Roadside Road -> Roadside Rd
Baker Road, London Road -> Baker Rd, London Rd


In [12]:
def words_of_length(text):
    return re.findall(r"\b\w{4,}\b", text)


for s in ["The quick brown fox jumps over the lazy dog", "I am ok"]:
    print(words_of_length(s))

['quick', 'brown', 'jumps', 'over', 'lazy']
[]


In [13]:
def keep_alnum(text):
    return re.sub(r"[^A-Za-z0-9]", "", text)


for s in ["abc123!@#", "a b-c_d", "中文abc"]:
    print(repr(s), "->", keep_alnum(s))

'abc123!@#' -> abc123
'a b-c_d' -> abcd
'中文abc' -> abc


In [14]:
def drop_parentheses(text):
    return re.sub(r"\s*\([^)]*\)", "", text)


for s in ["example (.com)", "w3resource", "github (.com)", "stackoverflow (.com)"]:
    print(drop_parentheses(s))

example
w3resource
github
stackoverflow


## 二、从网页检索气候数据



In [15]:
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

resp = requests.get("https://www.weather2travel.com/china/harbin/climate/",
                    headers=HEADERS, timeout=20)
print(resp.status_code, resp.encoding, len(resp.text))

soup = BeautifulSoup(resp.text, "html.parser")
tables = soup.find_all("table", class_="climate_table")
print("页面里的 table 数量：", len(soup.find_all("table")))
print("气候表格数量：", len(tables))

200 UTF-8 65214
页面里的 table 数量： 4
气候表格数量： 3


In [16]:
table = tables[0]

print("标题：", soup.find("h1").get_text(strip=True))
print("caption：", table.find("caption"))
print("直接子节点：", [(c.name, len(c.find_all("tr", recursive=False)))
                       for c in table.find_all(recursive=False)])
print("表头：", [c.get_text(strip=True) for c in table.find("tr").find_all(["th", "td"])])
first_row = table.find("tbody").find("tr")
print("第一行：", [c.get_text(" ", strip=True) for c in first_row.find_all(["th", "td"])][:3], "...")

标题： Harbin weather by month
caption： None
直接子节点： [('thead', 1), ('tbody', 3)]
表头： ['', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
第一行： ['Max temperature (daytime)', '-13', '-9'] ...


In [17]:
def retrieve_climate_table(s):
    resp = requests.get(s, headers=HEADERS, timeout=20)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    for table in soup.find_all("table"):
        rows = table.find_all("tr")[:3]
        text = " ".join(row.get_text(" ", strip=True) for row in rows)
        if sum(1 for m in MONTHS if m in text) >= 10:
            return table
    return None

In [18]:
cities = {
    "Harbin": "https://www.weather2travel.com/china/harbin/climate/",
    "Boston": "https://www.weather2travel.com/massachusetts/boston/climate/",
    "Miami": "https://www.weather2travel.com/florida/miami/climate/",
    "Madison": "https://www.weather2travel.com/wisconsin/madison/climate/",
}

for name, url in cities.items():
    table = retrieve_climate_table(url)
    print(f"{name}: {type(table).__name__}, {len(table.find_all('tr'))} 行")

Harbin: Tag, 4 行


Boston: Tag, 5 行


Miami: Tag, 5 行


Madison: Tag, 4 行


In [19]:
print("首页：", retrieve_climate_table("https://www.weather2travel.com/"))

try:
    retrieve_climate_table("https://www.weather2travel.com/no-such-page-xyz/")
except requests.HTTPError as e:
    print("404 抛出异常：", e)

首页： None


404 抛出异常： 404 Client Error: Not Found for url: https://www.weather2travel.com/no-such-page-xyz/


In [20]:
def list_climate_table_row_names(s):
    table = retrieve_climate_table(s)
    if table is None:
        return None

    names = []
    for tr in table.find_all("tr"):
        th = tr.find("th")
        if th is None or th.get("colspan"):
            continue
        name = th.get_text(" ", strip=True)
        if not name or name in MONTHS or name.startswith("Climate data"):
            continue
        names.append(name)
    return names

In [21]:
for name, url in cities.items():
    print(name, list_climate_table_row_names(url))

print("首页：", list_climate_table_row_names("https://www.weather2travel.com/"))

Harbin ['Max temperature (daytime)', 'Min temperature (night-time)', 'Heat & Humidity']


Boston ['Max temperature (daytime)', 'Min temperature (night-time)', 'Heat & Humidity', 'Sea temperature']


Miami ['Max temperature (daytime)', 'Min temperature (night-time)', 'Heat & Humidity', 'Sea temperature']


Madison ['Max temperature (daytime)', 'Min temperature (night-time)', 'Heat & Humidity']


首页： None
